# 04 - Entrenamiento secuencial B2 → B3 → B4

Este cuaderno ejecuta en orden la comprobación de las finalistas de B2 (`C=1.0` y `C=0.1` en A, B y C), una búsqueda acotada de B3 y una búsqueda acotada de B4. Al terminar una fase comienza automáticamente la siguiente. Usa todas las partidas de `train` en A, B y C; `val` solo compara finalistas y `test` no se toca.

Los estudios Optuna activos se escriben en el disco local de Colab y se respaldan de manera consistente en Drive tras cada pliegue y trial. En B2, cada combinación pliegue/`C` guarda sus métricas al terminar; B3/B4 hacen lo mismo por pliegue, de modo que una interrupción recupera los pliegues ya terminados del trial. `pipeline_status.json` salta las fases completas. La búsqueda eficiente v2 queda separada de los estudios lentos anteriores, para no mezclar protocolos.


In [ ]:
from pathlib import Path
import sys
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
# Carga el código actualizado después de instalar dependencias.
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa la descarga anterior.'
from ptcg_wp.pipeline import run_sequential_pipeline
from sklearn import __version__ as sklearn_version
assert tuple(map(int, sklearn_version.split('.')[:2])) >= (1, 7), 'B4 necesita scikit-learn >=1.7; reinicia el runtime tras instalar.'
print('Entorno listo | sklearn:', sklearn_version, '| Drive:', CHECKPOINT_ROOT)

In [ ]:
# Corrida completa y secuencial; cada fase conserva sus propios checkpoints.
# Si una operación lleva más de 10 min, Colab imprimirá dónde está (fit, evaluación o I/O).
import faulthandler
faulthandler.enable()
faulthandler.dump_traceback_later(600, repeat=True)
try:
    pipeline_outputs = run_sequential_pipeline(
    data_dir=DATA_DIR,
    checkpoint_root=CHECKPOINT_ROOT,
    batch_size=32768,
    b2_max_epochs=600,  # Doble del presupuesto de 300 de la primera búsqueda.
    b2_eta0=0.05,
    b2_gradient_tol=1e-3,
    b3_trials=5,  # 3 iniciales + 2 guiadas por TPE; espacio acotado.
    b3_n_jobs=2,
    b4_trials=5,
    b4_max_iter=500,
    b4_patience=30,
    b4_timeout=3600,  # Por métrica; Optuna termina tras el trial actual.
    max_ram_gib=10.0,
    seed=2026,
    verbose=True,
    )
finally:
    faulthandler.cancel_dump_traceback_later()
print('Resultados/checkpoints por fase:')
for fase, ruta in pipeline_outputs.items():
    print(f'{fase}: {ruta}')